# Library

In [1]:
import pandas as pd
import re
import numpy as np


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import time
import pickle

import warnings
warnings.filterwarnings('ignore')

import os
import string
from styleframe import StyleFrame
from IPython.display import display, HTML

def pretty_print(df):
    return display( HTML( df.to_html().replace("\\n","<br>") ) )

import statsmodels.api as sm 
from statsmodels.iolib.summary2 import summary_col
from scipy import stats

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
    

In [9]:
# # Example data
# df = pd.DataFrame({
#     "y": [1, 2, 3, 4, 5],
#     "x1": [2, 1, 3, 5, 4],
#     "x2": [5, 3, 4, 2, 1]
# })

# # Define dependent and independent variables
# Y = df["y"]
# X = df[["x1", "x2"]]

# # Add constant (intercept)
# X = sm.add_constant(X)

# # Run OLS
# model = sm.OLS(Y, X).fit()

# # Print results
# print(model.summary())



# Import data and adjust

## Import

In [13]:
dfmain = pd.read_csv("FedSpeechesRobust.csv", parse_dates = ['date'])
print(dfmain.shape)
dfmain.head()


(7177, 1044)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,SaltFresh_Other,Age,yearsIn,Chair,timeOnly_dt,hour_frac,TimeOfDay,tod_evening,tod_midday,tod_morning
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,0,NaN,1.0,0,NaN,NaN,NaN,0,0,0
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0


In [8]:
speechVarList = ['abstract','info','read','disunity','strain','strainUpper']

finTypeList = ["SPY","SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]

finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList1 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']

macroVarGTList2 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1',
                  'gIPB1', 'gIPF0', 'gIPF1']

topicList = ['community_development','monetary_policy_inflation','real_economy_productivity',
             'fed_operations_payments','financial_stability_regulation','international_economics']

otherList = ['is_monday', 'is_friday','Chair','tod_midday','tod_morning']+topicList

speakerBG = ['Gender', 'Race', 'Major_Law', 'Major_Business', 'Major_Fin', 'Major_Other', 'Degree_Other', 
             'PreFed_clean_AcademicEcon', 'PreFed_clean_Finance', 'PreFed_clean_Business', 'PreFed_clean_Law', 
             'PreFed_clean_Other', 'Freshwater', 'SaltFresh_Other', 'Age', 'yearsIn']


## IMPORTANT: subsample 

In [11]:
backup = dfmain.copy()
print(backup.shape)

(7177, 1044)


In [12]:
dfmain = backup.copy()
dfmain = dfmain[dfmain['speaker'] != 'Alan Greenspan']
print(dfmain.shape)


(6683, 1044)


# Functions

In [19]:
def OLSFin(df, finName, finVar, outputList, controlList, Ftest = True, sepNull = False, lagn = 4, feffect = "individual"):
    '''
    feffect: role - using IsBpres; institution - using District; 
    individual - using speaker; none - not using any
    '''
    resultReg = []
    inputlist = [finVar+"_"+finName+"_lag"+str(i+1) for i in range(lagn)]+controlList
    if  feffect == "role":
        df["IsBpres"] = df["IsBpres"].astype(int)
        exog = sm.add_constant(df[inputlist+["IsBpres"]])
        showvar = inputlist+["IsBpres"]
    elif feffect == "institution":
        desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                         'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                         'Kansas', 'Minneapolis']
        df["District"] = pd.Categorical(df["District"], categories=desired_order)
        dummies = pd.get_dummies(df["District"], prefix="inst", drop_first=True)
        dummies = dummies.astype(float)
        exog = sm.add_constant(pd.concat([df[inputlist], dummies], axis=1))
        showvar = inputlist + ["inst_"+name for name in desired_order[1:]]
    elif feffect == "individual":
        dummies = pd.get_dummies(df["speaker"], prefix="speaker", drop_first=True)
        dummies = dummies.astype(float)
        exog = sm.add_constant(pd.concat([df[inputlist], dummies], axis=1))
        showvar = inputlist
    elif feffect == "none":
        exog =  sm.add_constant(df[inputlist])
        showvar = inputlist
    else:
        print("category not recognized, choosing between role, institution, individual")
        return None
    
    for output in outputList:
        # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
        # print(exog.dtypes)
        reg = sm.OLS(df[output], exog, missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
        resultReg.append(reg)

    if not Ftest:
        # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
        resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                  model_names= outputList,
                                  regressor_order = showvar,
                                  drop_omitted = True,
                                  include_r2 = False,
                                  info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                      'N': lambda x: "{0:d}".format(int(x.nobs))})
        resultTable.tables[0].to_excel('results/robust/{}.xlsx'.format(finName+finVar))
        return resultTable
    elif sepNull:
        inputlist = inputlist[:4]
        results = pd.DataFrame(index=[finVar+"_"+finName], columns=outputList)
        for (reg, output) in zip(resultReg, outputList):
            # Restriction matrix: each coefficient equals zero
            beta_hat = reg.params[inputlist].values
            V = reg.cov_params().loc[inputlist, inputlist].values
            R = np.eye(len(inputlist))   # 4x4 identity
            
            theta = R @ beta_hat                 # shape (4,)
            var_theta = R @ V @ R.T              # shape (4,4)
            
            # Wald statistic
            try:
                W = theta.T @ np.linalg.inv(var_theta) @ theta
            except:
                print(inputlist)
                results.loc[finVar+"_"+finName, output] = "error"
                continue
            dfree = len(inputlist)                  # 4 restrictions
            
            # Convert to F-statistic (OLS convention)
            F_stat = W / dfree
            p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)

            if p_value < 0.01:
                results.loc[finVar+"_"+finName, output] = "***"
            elif p_value < 0.05:
                results.loc[finVar+"_"+finName, output] = "**"
            elif p_value < 0.1:
                results.loc[finVar+"_"+finName, output] = "*"
            else:
                results.loc[finVar+"_"+finName, output] = 0
        return results
    else:
        inputlist = inputlist[:4]
        results = pd.DataFrame(index=[finVar+"_"+finName], columns=outputList)
        for (reg, output) in zip(resultReg, outputList):
            # i will be testing the first lagn variable coefficient jointly sum to 0
            # against the alternative that they are >0 and <0
            beta_hat = reg.params[inputlist].values
            V = reg.cov_params().loc[inputlist, inputlist].values
        
            L = np.ones(len(inputlist))  # weights for sum
            theta = L @ beta_hat
            var_theta = L @ V @ L.T
            se_theta = np.sqrt(var_theta)
        
            t_stat = theta / se_theta
            df_resid = reg.df_resid
        
            # One-sided p-values
            p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
            p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
            
            # Decision rule
            if theta > 0:
                if p_pos < 0.01:
                    results.loc[finVar+"_"+finName, output] = "+++"
                elif p_pos < 0.05:
                    results.loc[finVar+"_"+finName, output] = "++"
                elif p_pos < 0.1:
                    results.loc[finVar+"_"+finName, output] = "+"
                else:
                    results.loc[finVar+"_"+finName, output] = 0
            elif theta < 0:
                if p_neg < 0.01:
                    results.loc[finVar+"_"+finName, output] = "---"
                elif p_neg < 0.05:
                    results.loc[finVar+"_"+finName, output] = "--"
                elif p_neg < 0.1:
                    results.loc[finVar+"_"+finName, output] = "-"
                else:
                    results.loc[finVar+"_"+finName, output] = 0
            else:
                results.loc[finVar+"_"+finName, output] = 0

        return results


In [21]:
def OLSFin2(df, finNameList, finVarList, inputList, Ftest = True, sepNull = False):
    results = pd.DataFrame(index=finNameList, columns=finVarList)
    testList = inputList[:6]
    for finVar in finVarList:
        resultReg = []
        model_names = []
        for finName in finNameList:
            # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
            # print(exog.dtypes)
            output = finVar+"_"+finName+"_fut"+str(1)
            reg = sm.OLS(df[output], df[inputList], missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
            resultReg.append(reg)
            model_names.append(output)
        if not Ftest:
            # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
            resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                      model_names= model_names,
                                      regressor_order = inputList,
                                      drop_omitted = True,
                                      include_r2 = False,
                                      info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                          'N': lambda x: "{0:d}".format(int(x.nobs))})
            resultTable.tables[0].to_excel('results/reg2/{}.xlsx'.format(finVar))
            print(resultTable)
        elif sepNull:
            for (reg,finName) in zip(resultReg,finNameList):
                # Restriction matrix: each coefficient equals zero
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
                R = np.eye(len(testList))   # 4x4 identity
                
                theta = R @ beta_hat                 # shape (4,)
                var_theta = R @ V @ R.T              # shape (4,4)
                
                # Wald statistic
                try:
                    W = theta.T @ np.linalg.inv(var_theta) @ theta
                except:
                    print(testList)
                    results.loc[finName, finVar] = "error"
                    continue
                dfree = len(testList)                  # 4 restrictions
                
                # Convert to F-statistic (OLS convention)
                F_stat = W / dfree
                p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)
    
                if p_value < 0.01:
                    results.loc[finName, finVar] = "***"
                elif p_value < 0.05:
                    results.loc[finName, finVar] = "**"
                elif p_value < 0.1:
                    results.loc[finName, finVar] = "*"
                else:
                    results.loc[finName, finVar] = 0
        else:
            for (reg,finName) in zip(resultReg,finNameList):
                # i will be testing the first lagn variable coefficient jointly sum to 0
                # against the alternative that they are >0 and <0
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
            
                L = np.ones(len(testList))  # weights for sum
                theta = L @ beta_hat
                var_theta = L @ V @ L.T
                se_theta = np.sqrt(var_theta)
            
                t_stat = theta / se_theta
                df_resid = reg.df_resid
            
                # One-sided p-values
                p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
                p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
                
                # Decision rule
                if theta > 0:
                    if p_pos < 0.01:
                        results.loc[finName, finVar] = "+++"
                    elif p_pos < 0.05:
                        results.loc[finName, finVar] = "++"
                    elif p_pos < 0.1:
                        results.loc[finName, finVar] = "+"
                    else:
                        results.loc[finName, finVar] = 0
                elif theta < 0:
                    if p_neg < 0.01:
                        results.loc[finName, finVar] = "---"
                    elif p_neg < 0.05:
                        results.loc[finName, finVar] = "--"
                    elif p_neg < 0.1:
                        results.loc[finName, finVar] = "-"
                    else:
                        results.loc[finName, finVar] = 0
                else:
                    results.loc[finName, finVar] = 0
    return results

In [23]:
def OLSFin3(df, finNameList, finVarList, inputList, windowList, windowAsym = True, Ftest = True, sepNull = False):
    finVarFullList = [i+str(j) for i in finVarList for j in windowList]
    results = pd.DataFrame(index=finNameList, columns=finVarFullList)
    testList = inputList[:6]
    for finName in finNameList:
        resultReg = []
        model_names = []
        for finVar in finVarList:
            for win in windowList:
                # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
                # print(exog.dtypes)
                if windowAsym:
                    df["output"] = df[finName+"_"+finVar+"_post_"+str(win)]
                else:
                    df["output"] = df[finName+"_"+finVar+"_reaction_"+str(win)]
                reg = sm.OLS(df["output"], df[inputList], missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
                resultReg.append(reg)
                model_names.append(finName+"_"+finVar+str(win))
        if not Ftest:
            # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
            resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                      model_names= model_names,
                                      regressor_order = inputList,
                                      drop_omitted = True,
                                      include_r2 = False,
                                      info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                          'N': lambda x: "{0:d}".format(int(x.nobs))})
            resultTable.tables[0].to_excel('results/robust/{}.xlsx'.format(finName))
            print(resultTable)
        elif sepNull:
            for (reg,finVarFull) in zip(resultReg,finVarFullList):
                # Restriction matrix: each coefficient equals zero
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
                R = np.eye(len(testList))   # 4x4 identity
                
                theta = R @ beta_hat                 # shape (4,)
                var_theta = R @ V @ R.T              # shape (4,4)
                
                # Wald statistic
                try:
                    W = theta.T @ np.linalg.inv(var_theta) @ theta
                except:
                    print(testList)
                    results.loc[finName, finVarFull] = "error"
                    continue
                dfree = len(testList)                  # 4 restrictions
                
                # Convert to F-statistic (OLS convention)
                F_stat = W / dfree
                p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)
    
                if p_value < 0.01:
                    results.loc[finName, finVarFull] = "***"
                elif p_value < 0.05:
                    results.loc[finName, finVarFull] = "**"
                elif p_value < 0.1:
                    results.loc[finName, finVarFull] = "*"
                else:
                    results.loc[finName, finVarFull] = 0
        else:
            for (reg,finVarFull) in zip(resultReg,finVarFullList):
                # i will be testing the first lagn variable coefficient jointly sum to 0
                # against the alternative that they are >0 and <0
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
            
                L = np.ones(len(testList))  # weights for sum
                theta = L @ beta_hat
                var_theta = L @ V @ L.T
                se_theta = np.sqrt(var_theta)
            
                t_stat = theta / se_theta
                df_resid = reg.df_resid
            
                # One-sided p-values
                p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
                p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
                
                # Decision rule
                if theta > 0:
                    if p_pos < 0.01:
                        results.loc[finName, finVarFull] = "+++"
                    elif p_pos < 0.05:
                        results.loc[finName, finVarFull] = "++"
                    elif p_pos < 0.1:
                        results.loc[finName, finVarFull] = "+"
                    else:
                        results.loc[finName, finVarFull] = 0
                elif theta < 0:
                    if p_neg < 0.01:
                        results.loc[finName, finVarFull] = "---"
                    elif p_neg < 0.05:
                        results.loc[finName, finVarFull] = "--"
                    elif p_neg < 0.1:
                        results.loc[finName, finVarFull] = "-"
                    else:
                        results.loc[finName, finVarFull] = 0
                else:
                    results.loc[finName, finVarFull] = 0
    return results

In [25]:
def df_to_latex(df):
    def fmt_col(col):
        parts = str(col).replace("_", " ").split()
        s = "\\makecell{" + " \\\\ ".join(parts) + "}"
        # escape braces for Python's .format, but LaTeX will still see single braces
        s = s.replace("{", "{{").replace("}", "}}")
        return s

    latex = df.to_latex(
        index=True,
        escape=False,
        column_format="l" + "r" * len(df.columns),
        header=[fmt_col(c) for c in df.columns],
        float_format="%.3f",
    )
    return latex

def df_to_latex_intra(df):
    header = """
    \\begin{tabular}{lcccccccccccccccc}
    \\toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\\\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\\\
    \midrule
    """
    footer = r"""
    \bottomrule
    \end{tabular}
    """
    
    latex = df.to_latex(
    index=True,
    escape=False,
    column_format="l" + "r" * len(df.columns),
    header=False,   # <-- IMPORTANT
    float_format="%.3f",
    )
    
    # Remove the \begin{tabular} ... \end{tabular} wrapper
    body = latex.split("\\midrule")[1]          # everything after \midrule
    body = body.rsplit("\\bottomrule", 1)[0]    # everything before \bottomrule
    full_table = header + body + footer
    return full_table



In [27]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

def anova_group_ss(df, yvar, speaker_vars, financial_vars, macro_vars, topics = False, topic_vars = []):
    # Build formula
    if topics:
        all_vars = speaker_vars + financial_vars + macro_vars + topic_vars
    else:
        all_vars = speaker_vars + financial_vars + macro_vars
    formula = yvar + " ~ " + " + ".join(all_vars)
    
    # Fit model
    model = ols(formula, data=df).fit()
    
    # ANOVA
    anova_res = anova_lm(model, typ=2)
    
    # Extract group sums of squares
    speaker_ss = anova_res.loc[speaker_vars, 'sum_sq'].sum()
    financial_ss = anova_res.loc[financial_vars, 'sum_sq'].sum()
    macro_ss = anova_res.loc[macro_vars, 'sum_sq'].sum()
    if topics:
        topic_ss = anova_res.loc[topic_vars, 'sum_sq'].sum()
    
    total_ss = anova_res['sum_sq'].sum()
    
    speaker_pct = f"{(speaker_ss / total_ss) * 100:.2f}\%"
    financial_pct = f"{(financial_ss / total_ss) * 100:.2}\%"
    macro_pct = f"{(macro_ss / total_ss) * 100:.2f}\%"
    if topics:
        topic_pct = f"{(topic_ss / total_ss) * 100:.2f}\%"
    
        return {
        'speaker_pct': speaker_pct,
        'financial_pct': financial_pct,
        'macro_pct': macro_pct,
        'topic_pct': topic_pct
        }
    else:
        return {
        'speaker_pct': speaker_pct,
        'financial_pct': financial_pct,
        'macro_pct': macro_pct
        }



# Regressions

## First group of regression

In [32]:
finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']


In [ ]:
# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
controlList = realAct3+inflaty2+unemp+mich+topicList

for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/robust/RegG1main.csv")
display(model)


In [38]:
name1 = ["& SPY","& ES","& SHY","& IEF","& TLT","& UUP","& VXX","& VIX","& GLD"]
name2 = ["Return","Absolute Return","High-low Range","Volume","Abnormal Return (5 days)","Abnormal Return (22 days)"]
nameFinal = [name2[0]+' '+name1[0]]+name1[1:]
for var2 in name2[1:]:
    nameFinal += [var2+name1[0]]+name1[1:]
model.index = nameFinal
latex = df_to_latex(model)   # or whatever object you pass in
print(latex)



\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{abstract} & \makecell{info} & \makecell{read} & \makecell{disunity} & \makecell{strain} & \makecell{strainUpper} \\
\midrule
Return & SPY & 0 & 0 & 0 & 0 & 0 & 0 \\
& ES & 0 & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & ** & 0 & 0 & 0 & 0 \\
& IEF & 0 & 0 & 0 & 0 & 0 & 0 \\
& TLT & 0 & 0 & 0 & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & * & 0 \\
& VXX & 0 & 0 & 0 & 0 & 0 & ** \\
& VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
& GLD & 0 & 0 & 0 & 0 & 0 & 0 \\
Absolute Return& SPY & 0 & 0 & 0 & 0 & 0 & ** \\
& ES & 0 & 0 & 0 & ** & 0 & ** \\
& SHY & 0 & 0 & 0 & 0 & ** & 0 \\
& IEF & ** & 0 & 0 & 0 & 0 & 0 \\
& TLT & *** & 0 & ** & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & 0 & 0 \\
& VXX & ** & ** & 0 & 0 & 0 & * \\
& VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
& GLD & 0 & 0 & *** & * & * & 0 \\
High-low Range& SPY & *** & 0 & 0 & 0 & 0 & 0 \\
& ES & ** & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & 0 & 0 & 0 & 0 & 0 \\
& IEF & ** & 0 & ** & 0 & 0 & 0 \\
& TLT & ** & 0 & *** & 0 & 0 & 0 \\
& UUP & * & 0 

In [38]:
# using F-test but testing if individual coefficients are significant
outputList = speechVarList
controlList = realAct3+inflaty2+unemp+mich+topicList

for finVar in finVarList:
    for finType in finTypeList:
        model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = False, sepNull=False, feffect = "individual")

            

## Sub-reg of First group

In [42]:
outputList = speechVarList
controlList = speakerBG
controlList = controlList+ realAct3 +inflaty2 + mich + unemp + topicList

finVar = "volume"
finType = "SPX_index"
model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "role")
model.tables[0].to_excel(f"results/robust/RegG1mainSub{finVar+finType}.xlsx")
display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00**,0.00,0.00*,0.00***,0.00**
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,-0.00,-0.00,0.00,0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,0.00,-0.00,0.00**,0.00*,0.00**,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00*,0.00,-0.00,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.34***,-0.22***,-0.20***,-0.15***,-0.22***,0.05
,(0.08),(0.03),(0.02),(0.04),(0.03),(0.03)


In [43]:
columnName = ["Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
indexName = ["fin1","fin2","fin3","fin4","Gender","Race","Major: Law","Major: Business","Major: Finance","Major: Other",
             "Not PhD","Pre-Fed: Academic","Pre-Fed: Finance","Pre-Fed: Business","Pre-Fed: Law","Pre-Fed: Other",
             "Freshwater","Other Economist","Age","Terms Time","gINDPRO","gINDPRO lag1","Infl",
             "Infl lag1","MICH","MICH lag1","Unemp","Unemp lag1","Topic: Community Dev","Topic: MonPol–Inflation","Topic: Real Econ–Prod",
             "Topic: Fed Ops–Payments","Topic: FinStab–Reg","Topic: International Econ", "Is Bank Pres","N","$R^2_{adj}$"]
res = model.tables[0]
res.columns = columnName
res.index = [elem for pair in zip(indexName[:-2], [""] * len(indexName[:-2])) for elem in pair][:-1]+[""]+indexName[-2:]
latex = df_to_latex(res)   # or whatever object you pass in
print(latex)


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
fin1 & -0.00 & 0.00** & 0.00 & 0.00* & 0.00*** & 0.00** \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin2 & -0.00 & -0.00 & 0.00 & 0.00 & -0.00 & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin3 & 0.00 & -0.00 & 0.00** & 0.00* & 0.00** & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin4 & -0.00* & 0.00 & -0.00 & 0.00 & 0.00 & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
Gender & -0.34*** & -0.22*** & -0.20*** & -0.15*** & -0.22*** & 0.05 \\
 & (0.08) & (0.03) & (0.02) & (0.04) & (0.03) & (0.03) \\
Race & 0.26* & 0.06 & 0.37*** & 0.34*** & 0.18*** & 0.49*** \\
 & (0.14) & (0.06) & (0.04) & (0.06) & (0.05) & (0.05) \\
Major: Law & 0.19 & 0.10 & -0.12** & -0.25*** & -0.00 & 0.11* \\
 & (0.16) & (0.07) & (0.05) & 

## ANOVA


In [46]:
# Combine all regressors
macro_vars = realAct1+realAct2+realAct3+inflat1+inflat2+inflaty1+inflaty2+unemp+mich # macroVarGTList
if not "inst_NewYork" in dfmain.columns:
    desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                     'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                     'Kansas', 'Minneapolis']
    dfmain["District"] = pd.Categorical(dfmain["District"], categories=desired_order)
    dummies = pd.get_dummies(dfmain["District"], prefix="inst", drop_first=True).astype(int)
    dfmain = pd.concat([dfmain, dummies], axis=1)
speaker_vars = speakerBG + ["inst_"+var for var in desired_order[1:]]

finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

anova_tables = {}
for finType in finTypeList: 
    financial_vars = []
    for finVar in finVarList: 
        financial_vars = financial_vars+[finVar+"_"+finType+"_lag"+str(i+1) for i in range(4)]
    results = {}
    for yvar in speechVarList:
        results[yvar] = anova_group_ss(dfmain, yvar, speaker_vars, financial_vars, macro_vars, topics=True, topic_vars=topicList)
    anova_tables[finType] = pd.DataFrame(results).reset_index().rename(columns={'index': 'group'})

for name, tbl in anova_tables.items():
    tbl['equity'] = name
big_table = pd.concat(anova_tables.values(), axis=0)
big_table.index = big_table['equity']
big_table = big_table.drop(columns = ['equity'])
big_table.to_csv("results/robust/RegG1Anova1.csv")
big_table


,group,abstract,info,read,disunity,strain,strainUpper
equity,,,,,,,
SPX_index,speaker_pct,4.19\%,5.97\%,20.12\%,18.64\%,24.27\%,13.95\%
SPX_index,financial_pct,0.72\%,0.14\%,0.14\%,0.22\%,0.21\%,0.35\%
SPX_index,macro_pct,0.56\%,0.55\%,0.29\%,0.26\%,0.21\%,0.53\%
SPX_index,topic_pct,5.34\%,1.84\%,36.81\%,22.48\%,3.64\%,7.11\%
ES_futures,speaker_pct,6.83\%,11.16\%,20.23\%,15.46\%,28.99\%,18.11\%
ES_futures,financial_pct,0.51\%,0.9\%,0.13\%,0.32\%,0.29\%,0.72\%
ES_futures,macro_pct,0.96\%,1.04\%,0.49\%,0.28\%,0.71\%,1.00\%
ES_futures,topic_pct,4.72\%,2.06\%,39.56\%,25.07\%,2.67\%,6.44\%
SHY,speaker_pct,6.39\%,12.13\%,18.88\%,12.76\%,25.86\%,17.87\%


In [47]:
nameList = ['speaker','finance','macro','topics']
big_table['group'] = big_table['group'].map(dict(zip(big_table['group'].value_counts().index.tolist(),nameList)))
nameList = ["SPY","ES"]+big_table.index.value_counts().index.tolist()[2:]
nameTotal = []
for var in nameList:
    nameTotal += [var]+[""]*3
big_table.index = nameTotal
columnName = ["Group","Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
big_table.columns = columnName
latex = df_to_latex(big_table)
print(latex)


\begin{tabular}{lrrrrrrr}
\toprule
 & \makecell{Group} & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
SPY & speaker & 4.19\% & 5.97\% & 20.12\% & 18.64\% & 24.27\% & 13.95\% \\
 & finance & 0.72\% & 0.14\% & 0.14\% & 0.22\% & 0.21\% & 0.35\% \\
 & macro & 0.56\% & 0.55\% & 0.29\% & 0.26\% & 0.21\% & 0.53\% \\
 & topics & 5.34\% & 1.84\% & 36.81\% & 22.48\% & 3.64\% & 7.11\% \\
ES & speaker & 6.83\% & 11.16\% & 20.23\% & 15.46\% & 28.99\% & 18.11\% \\
 & finance & 0.51\% & 0.9\% & 0.13\% & 0.32\% & 0.29\% & 0.72\% \\
 & macro & 0.96\% & 1.04\% & 0.49\% & 0.28\% & 0.71\% & 1.00\% \\
 & topics & 4.72\% & 2.06\% & 39.56\% & 25.07\% & 2.67\% & 6.44\% \\
SHY & speaker & 6.39\% & 12.13\% & 18.88\% & 12.76\% & 25.86\% & 17.87\% \\
 & finance & 1.1\% & 0.82\% & 0.4\% & 0.47\% & 0.6\% & 0.82\% \\
 & macro & 1.09\% & 0.68\% & 0.62\% & 0.41\% & 0.88\% & 1.10\% \\
 & topics & 4.80\

## Second group of regression

In [50]:
finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']
             
otherList = ['is_monday', 'is_friday','Chair']+topicList


In [51]:
inputList = speechVarList+realAct3+inflaty2+unemp+mich+otherList
finNameList = finTypeList
indexName = ["SPX","ES","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
columnName = ["Return","Absolute Return","High-low Range","Volume","AbnormalReturn (5days)","AbnormalReturn (22days)"]

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/robust/RegG2main.csv")
res.index = indexName
res.columns = columnName
latex = df_to_latex(res)
display(res)
print(latex)

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True)
res.to_csv("results/robust/RegG2robustSign.csv")
res.index = indexName
res.columns = columnName
latex = df_to_latex(res)
display(res)
print(latex)



['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,Return,Absolute Return,High-low Range,Volume,AbnormalReturn (5days),AbnormalReturn (22days)
SPX,0,***,***,***,0,0
ES,0,0,0,***,0,0
SHY,0,***,***,***,0,0
IEF,*,***,**,***,0,0
TLT,0,***,***,***,0,0
UUP,0,0,**,0,0,0
VXX,0,0,*,***,0,0
VIX,0,*,***,error,0,0
GLD,0,0,*,***,0,0


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Return} & \makecell{Absolute \\ Return} & \makecell{High-low \\ Range} & \makecell{Volume} & \makecell{AbnormalReturn \\ (5days)} & \makecell{AbnormalReturn \\ (22days)} \\
\midrule
SPX & 0 & *** & *** & *** & 0 & 0 \\
ES & 0 & 0 & 0 & *** & 0 & 0 \\
SHY & 0 & *** & *** & *** & 0 & 0 \\
IEF & * & *** & ** & *** & 0 & 0 \\
TLT & 0 & *** & *** & *** & 0 & 0 \\
UUP & 0 & 0 & ** & 0 & 0 & 0 \\
VXX & 0 & 0 & * & *** & 0 & 0 \\
VIX & 0 & * & *** & error & 0 & 0 \\
GLD & 0 & 0 & * & *** & 0 & 0 \\
\bottomrule
\end{tabular}



,Return,Absolute Return,High-low Range,Volume,AbnormalReturn (5days),AbnormalReturn (22days)
SPX,0,-,--,+++,0,0
ES,0,--,--,-,0,0
SHY,0,--,0,---,0,0
IEF,0,---,---,---,0,0
TLT,0,---,---,---,0,0
UUP,0,0,0,--,0,0
VXX,0,-,--,+,0,0
VIX,0,-,--,0,0,0
GLD,0,-,0,---,0,0


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Return} & \makecell{Absolute \\ Return} & \makecell{High-low \\ Range} & \makecell{Volume} & \makecell{AbnormalReturn \\ (5days)} & \makecell{AbnormalReturn \\ (22days)} \\
\midrule
SPX & 0 & - & -- & +++ & 0 & 0 \\
ES & 0 & -- & -- & - & 0 & 0 \\
SHY & 0 & -- & 0 & --- & 0 & 0 \\
IEF & 0 & --- & --- & --- & 0 & 0 \\
TLT & 0 & --- & --- & --- & 0 & 0 \\
UUP & 0 & 0 & 0 & -- & 0 & 0 \\
VXX & 0 & - & -- & + & 0 & 0 \\
VIX & 0 & - & -- & 0 & 0 & 0 \\
GLD & 0 & - & 0 & --- & 0 & 0 \\
\bottomrule
\end{tabular}



## Third group of regression

In [39]:
windowList = [15,30,45,60]
finTypeList = ["SPY_BB","SHY_BB", "IEF_BB", "TLT_BB","VIX_BB"]
finVarList = ["ret", "absret", "range","vol"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']
             
otherList = ['is_monday', 'is_friday','Chair','tod_midday','tod_morning']+topicList


In [40]:
finNameList = finTypeList[:-1]
inputList = speechVarList+otherList #+realAct3+inflaty2+unemp+mich
indexName = ["SPY","SHY","IEF","TLT","VIX"]

res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = False)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = False)
res = pd.concat([res1,res2])
res.to_csv("results/robust/RegG3robustSign.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)

res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = True)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = True)
res = pd.concat([res1,res2])
res.to_csv("results/robust/RegG3main.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)



,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,0,+,0,0,0,--,0,---,--,---
SHY,0,0,--,-,0,0,0,0,0,0,+,0,0,--,--,---
IEF,0,0,---,0,+++,++,++,++,++,0,0,0,0,---,--,--
TLT,0,0,--,0,+++,++,0,0,+++,++,++,0,0,--,0,---
VIX,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & 0 & + & 0 & 0 & 0 & -- & 0 & --- & -- & --- \\
SHY & 0 & 0 & -- & - & 0 & 0 & 0 & 0 & 0 & 0 & + & 0 & 0 & -- & -- & --- \\
IEF & 0 & 0 & --- & 0 & +++ & ++ & ++ & ++ & ++ & 0 & 0 & 0 & 0 & --- & -- & -- \\
TLT & 0 & 0 & -- & 0 & +++ & ++ & 0 & 0 & +++ & ++ & ++ & 0 & 0 & -- & 0 & --- \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,***,0,0,0,*,**,0,***,**,***
SHY,0,0,*,0,*,0,0,0,**,0,0,0,0,**,0,**
IEF,0,0,0,0,***,0,0,*,**,0,**,0,0,***,**,***
TLT,0,0,0,0,*,0,0,0,*,0,**,0,0,**,0,***
VIX,0,0,0,0,0,0,0,*,0,**,0,0,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & *** & 0 & 0 & 0 & * & ** & 0 & *** & ** & *** \\
SHY & 0 & 0 & * & 0 & * & 0 & 0 & 0 & ** & 0 & 0 & 0 & 0 & ** & 0 & ** \\
IEF & 0 & 0 & 0 & 0 & *** & 0 & 0 & * & ** & 0 & ** & 0 & 0 & *** & ** & *** \\
TLT & 0 & 0 & 0 & 0 & * & 0 & 0 & 0 & * & 0 & ** & 0 & 0 & ** & 0 & *** \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 & 0 & * & 0 & ** & 0 & 0 & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


In [41]:
# the details
res = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = False, sepNull = False)

res = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = False, sepNull = False)



                               SPY_BB_ret15 SPY_BB_ret30 SPY_BB_ret45 SPY_BB_ret60 SPY_BB_absret15 SPY_BB_absret30 SPY_BB_absret45 SPY_BB_absret60 SPY_BB_range15 SPY_BB_range30 SPY_BB_range45 SPY_BB_range60  SPY_BB_vol15   SPY_BB_vol30   SPY_BB_vol45   SPY_BB_vol60 
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
abstract                       -0.00        -0.00        -0.00        -0.00*       -0.00           -0.00*          -0.00           -0.00           -0.01          -0.01          0.00           -0.01          -67533.39**    -71816.50*     -18164.40      20130.56      
                               (0.00)       (0.00)       (0.00)       (0.00)       (0.00)          (0.00)          (0.00)          (0.00)          (0.01)         (0.01)         (0.01)         (0.01)

# Making to be Power BI datasets


In [2]:
dfmain = pd.read_csv("FedSpeechesRobust.csv", parse_dates = ['date'])
print(dfmain.shape)
dfmain.head()


(7177, 1044)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,SaltFresh_Other,Age,yearsIn,Chair,timeOnly_dt,hour_frac,TimeOfDay,tod_evening,tod_midday,tod_morning
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,0,NaN,1.0,0,NaN,NaN,NaN,0,0,0
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,0,55.0,3.0,0,NaN,NaN,NaN,0,0,0


In [7]:
df = dfmain.copy()
macroList = [
    'UNRATE','CPIAUCSL_yoy','PCEPILFE_yoy','INDPRO',
    'MICH','GDPC1_change','GDPC1_low_q10'
]

finList = [
    'ret_SPY','hl_range_SPY','volume_SPY',
    'ret_IEF','hl_range_IEF','volume_IEF',
    'ret_VIX'
]

topicList = [
    'fed_operations_payments', 'international_economics',
    'financial_stability_regulation', 'real_economy_productivity',
    'monetary_policy_inflation', 'community_development'
]

otherList = [
    'District', 'title', 'speaker', 'location', 'date',
    'board_role', 'IsBpres', 'timeOnly',
    'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper',
    'is_monday', 'is_friday',
    'Gender', 'Birth Year', 'Start Year', 'End Year', 'Race',
    'MajorCollapsed', 'DegreeCollapsed', 'PreFed_cleanCollapsed',
    'Saltwater', 'Freshwater',
    'Age', 'yearsIn', 'TimeOfDay'
]

df['speaker_id'] = df['speaker'].astype('category').cat.codes
df['institution_id'] = df['District'].astype('category').cat.codes
df['speech_id'] = df.reset_index().index


In [9]:
# the center of the star
fact_cols = (
    ['speech_id', 'speaker_id', 'institution_id', 'date', 'title', 'location',
     'timeOnly', 'TimeOfDay', 'Age', 'yearsIn', 'is_monday', 'is_friday'] +
    topicList +
    ['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper',
     'board_role', 'IsBpres']
)
fact_speech_metrics = df[fact_cols].copy()

rename_map = {
    'timeOnly': 'SpeechTime',
    'TimeOfDay': 'SpeechTimeOfDay',
    'abstract': 'Complexity_Abstract',
    'info': 'Complexity_Info',
    'read': 'Complexity_Readability',
    'disunity': 'Complexity_Disunity',
    'strain': 'Complexity_Strain',
    'strainUpper': 'Complexity_StrainUpper'
}
fact_speech_metrics.rename(columns = rename_map, inplace = True)

# speaker dimension
speaker_cols = [
    'speaker_id', 'speaker', 'Gender', 'Birth Year', 'Race',
    'MajorCollapsed', 'DegreeCollapsed', 'PreFed_cleanCollapsed',
    'Saltwater', 'Freshwater', 'Start Year', 'End Year'
]

dim_speaker = df[speaker_cols].drop_duplicates(subset='speaker_id').copy()
rename_map = {
    'MajorCollapsed': 'Major',
    'DegreeCollapsed': 'Degree',
    'PreFed_cleanCollapsed': 'PreFedBackground'
}
dim_speaker.rename(columns = rename_map, inplace = True)

# institution dimension
inst_cols = ['institution_id', 'District']

dim_institution = df[inst_cols].drop_duplicates(subset='institution_id').copy()

# date dimension
dim_date = df[['date']].drop_duplicates().copy()

dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['month'] = dim_date['date'].dt.month
dim_date['day_of_week'] = dim_date['date'].dt.day_name()

# macro 
macro_cols = ['date'] + macroList

dim_macro_daily = df[macro_cols].drop_duplicates(subset='date').copy()

# fin
fin_cols = ['date'] + finList

dim_financial_daily = df[fin_cols].drop_duplicates(subset='date').copy()


In [11]:
# export
# Create a single Excel writer
with pd.ExcelWriter("PowerBI/FedSpeechesData.xlsx", engine="xlsxwriter") as writer:
    
    fact_speech_metrics.to_excel(writer, sheet_name="fact_speech_metrics", index=False)
    dim_speaker.to_excel(writer, sheet_name="dim_speaker", index=False)
    dim_institution.to_excel(writer, sheet_name="dim_institution", index=False)
    dim_date.to_excel(writer, sheet_name="dim_date", index=False)
    dim_macro_daily.to_excel(writer, sheet_name="dim_macro_daily", index=False)
    dim_financial_daily.to_excel(writer, sheet_name="dim_financial_daily", index=False)

    # Optional: auto-adjust column widths for readability
    for sheet in writer.sheets:
        worksheet = writer.sheets[sheet]
        for idx, col in enumerate(eval(sheet).columns):
            max_len = max(
                eval(sheet)[col].astype(str).map(len).max(),
                len(col)
            ) + 2
            worksheet.set_column(idx, idx, max_len)



# Other things

In [33]:
df = pd.read_csv("FedSpeechesTopic.csv", parse_dates = ['date'])


In [35]:
#fixing up board governors names duplication and complications
# fixing up the data:
# Duplicate names
dupNameList = {
    # Powell
    "Jerome H Powell": "Jerome H. Powell",
    "Chair Jerome H. Powell": "Jerome H. Powell",

    # Bowman
    "Michelle W Bowman": "Michelle W. Bowman",
    "Governor Michelle W. Bowman": "Michelle W. Bowman",
    "Vice Chair for Supervision Michelle W. Bowman": "Michelle W. Bowman",

    # Waller
    "Christopher J Waller": "Christopher J. Waller",
    "Governor Christopher J. Waller": "Christopher J. Waller",

    # Cook
    "Lisa D Cook": "Lisa D. Cook",
    "Governor Lisa D. Cook": "Lisa D. Cook",

    # Barr
    "Michael S Barr": "Michael S. Barr",
    "Governor Michael S. Barr": "Michael S. Barr",
    "Vice Chair for Supervision Michael S. Barr": "Michael S. Barr",

    # Jefferson
    "Philip N Jefferson": "Philip N. Jefferson",
    "Vice Chair Philip N. Jefferson": "Philip N. Jefferson",

    # Lindsey
    "Lawrence Lindsey": "Lawrence B. Lindsey",
    "Lawrence B Lindsey": "Lawrence B. Lindsey",

    #
    "Janet L Yellen":"Janet L. Yellen",

    # Non-governor (flag)
    "Brian F Madigan": None,   # staff, not a governor

    # New York
        # --- Presidents (canonical forms) ---
    "William C Dudley": "William C. Dudley",
    "William J McDonough": "William J. McDonough",
    "John C Williams": "John C. Williams",
    "John C. Williams,": "John C. Williams",
    "John C. Williams, President and Chief Executive Officer": "John C. Williams",
    "John C. Williams, President and Chief Executive Officer ": "John C. Williams",

    # --- Timothy Geithner ---
    "Timothy F Geithner": "Timothy F. Geithner",

    # --- Simon Potter ---
    "Simon M Potter": "Simon M. Potter",
    "Simon Potter": "Simon M. Potter",

    # --- Nathaniel Wuerffel ---
    "NathanielWuerffel": "Nathaniel Wuerffel",
    "Nathaniel Wuerffel": "Nathaniel Wuerffel",

    # --- Michelle Neal ---
    "Michelle Neal": "Michelle Neal",
    "Michelle Neal, Head of the Markets Group": "Michelle Neal",

    # San Francisco
    "Mary C Daly": "Mary C. Daly",
    "Mary C. Daly": "Mary C. Daly",
    "Robert T Parry": "Robert T. Parry",
    "Janet L Yellen": "Janet L. Yellen",

    # Chicago
    "Michael Moskow": "Michael H. Moskow",
    "Michael H Moskow": "Michael H. Moskow",

    "Charles L Evans": "Charles L. Evans",

    "Austan D Goolsbee": "Austan D. Goolsbee",
    "Austan D. Goolsbee": "Austan D. Goolsbee",
    "Austan Goolsbee": "Austan D. Goolsbee",

    # Cleveland
    "Loretta J Mester": "Loretta J. Mester",
    "Loretta J. Mester": "Loretta J. Mester",
    "Jerry L Jordan": "Jerry L. Jordan",
    "W Lee Hoskins": "W. Lee Hoskins",
    "Beth M. Hammack": "Beth M. Hammack",

    # Philadelphia
    "Patrick T Harker": "Patrick T. Harker",
    "Patrick T. Harker": "Patrick T. Harker",
    "Charles I Plosser": "Charles I. Plosser",
    "Anthony M Santomero": "Anthony M. Santomero",
    "Edward G Boehne": "Edward G. Boehne",

    # St. Louis
    "Alberto Musalem": "Alberto G. Musalem",
    "Alberto G Musalem": "Alberto G. Musalem",
    "Alberto G. Musalem ": "Alberto G. Musalem",
    "Thomas C Melzer": "Thomas C. Melzer",

    # Boston
    "Susan M. Collins": "Susan M. Collins",
    "Susan M Collins": "Susan M. Collins",
    "By Susan M. Collins": "Susan M. Collins",
    "Eric S Rosengren": "Eric S. Rosengren",
    "Cathy E Minehan": "Cathy E. Minehan",
    "Paul M Connolly": "Paul M. Connolly",
    "Kenneth C Montgomery": "Kenneth C. Montgomery",
    "Lynn E Browne": "Lynn E. Browne",

    # Dallas
    "Lorie K. Logan": "Lorie K. Logan",
    "Lorie K Logan": "Lorie K. Logan",
    "Lorie Logan": "Lorie K. Logan",
    "Richard W Fisher": "Richard W. Fisher",
    "Robert D McTeer": "Robert D. McTeer",
    "Robert S Kaplan": "Robert S. Kaplan",

    # Kansas
    "Jeff Schmid": "Jeffrey Schmid",
    "Jeff Schmid ": "Jeffrey Schmid",
    "Jeffrey Schmid": "Jeffrey Schmid",
    "Thomas M Hoenig": "Thomas M. Hoenig",
    "Esther L George": "Esther L. George",

    # Richmond
    "J Alfred Broaddus":"J. Alfred Broaddus",
    "Jeffrey M Lacker":"Jeffrey M. Lacker",

    # Minneapolis
    "Gary H Stern":"Gary H. Stern",

    #
    "Robert P Forrestal":"Robert P. Forrestal",
    "Kartik Athreya":"Kartik B. Athreya"
}


df["speaker"] = df["speaker"].replace(dupNameList)


In [37]:
df.to_csv('FedSpeechesTopic.csv', date_format='%Y-%m-%d', index = False)
